# Final comparison — all models, synthetic test **and** real annotated trays

Loads every trained checkpoint **from its own training-output folder** and evaluates each on:

| | dataset | classes |
|---|---|---|
| **A** | synthetic test split | 23, held-out, specimen-disjoint |
| **B** | **3DTRAB_Detection_Annotated_v1** | single class (detection + localisation only) |
| **C** | 3DTRAB | the 5 orders actually annotated |

3DTRAB is **92 real tray photographs with 3,072 annotated insects** on our own imaging
setup, and no model has ever seen it — a genuinely held-out real benchmark.

**Each model is run the way it was trained** (whole-image models whole, tiled models
tiled + merge), which is the fair as-deployed comparison.

---

## Run order — two kernels

flatbug and RF-DETR live in different environments (different Python **and** different
torch majors), so a checkpoint is loaded by the same interpreter that wrote it.
Predictions are written to disk, so the two halves never need to coexist:

| step | cell | kernel |
|---|---|---|
| 1 | CONFIG → tiling → 3DTRAB prep → predictor → metrics | `Python (flatbug)` |
| 2 | **RUN A — flatbug models** | `Python (flatbug)` |
| 3 | *switch kernel*, re-run CONFIG → metrics | `rfdetr-env` |
| 4 | **RUN B — RF-DETR models** | `rfdetr-env` |
| 5 | **SCORE** + **Figures** | either (numpy / pandas / matplotlib only) |

Predictions are cached per model per dataset (`preds__{dataset}__{model}.json`), so every
cell is safe to re-run, resume after an interruption, or run in either order.

## Where each model comes from

| model | training notebook | output folder |
|---|---|---|
| flatbug whole @1536 | `train_flatbug_fullres.ipynb` | `sept_run/flatbug_whole_1536` |
| flatbug tiled @1536 | `train_flatbug_tiled_fullres.ipynb` | `sept_run/flatbug_tiled_1536` |
| RF-DETR whole @1536 | `train_rfdetr_fullres.ipynb` | `sept_run/rfdetr_whole_1536` |
| RF-DETR tiled @1536 | `train_rfdetr_tiled_fullres.ipynb` | `sept_run/rfdetr_tiled_1536` |

The CONFIG cell resolves the checkpoint **inside** each folder (`best.pt` for flatbug,
`checkpoint_best_total.pth` for RF-DETR) and prints exactly which file it picked, so the
table above is verified at run time rather than assumed. Folders that are absent are
listed and skipped — the comparison still runs on the rest.

> Two limits of B/C, worth stating in the meeting: 3DTRAB contains only **5** of the 23
> orders and is **85% Coleoptera**, so C is informative for those five only. `Aranea` and
> `Heteroptera` in the annotations are mapped to `Araneae` / `Hemiptera`.


In [ ]:
# ============================================================
# SEPTEMBER RUN — model comparison
# ============================================================
# Regenerated dataset, trained at full resolution. What changed since August:
#
#   * GROUND TRUTH boxes are the insect, not the rotated canvas. The old boxes were
#     1.295x too large on a side (4/pi for a uniform angle), so ~40% of every box was
#     tray. Every AP number from before was measured against inflated targets.
#   * TRAYS are 3473x2769 = 10.85 px/mm, matching the real 3DTRAB trays exactly.
#     Previously 2644x2108 = 8.26 px/mm, so synthetic insects were 24% too small.
#   * TAXONOMY is 22 classes, not 23. Siphonaptera and Raphidioptera dropped;
#     Diplopoda -> Polyxenida, Chilopoda -> Lithobiomorpha + Scutigeromorpha.
#   * DATA is synthetic + real 3DTRAB mixed, group-disjoint on the real side, with
#     the real half oversampled to ~5% of the training signal.
#
# Evaluate on 3dtrab_coco_TESTSPLIT.json, NEVER the full 3DTRAB — roughly half of
# those photographs are in this training set.

# ============================================================
# CONFIG — final comparison   <- edit paths/GPU here only
# ============================================================
import os
GPU_ID = "0"    # comparison runs alone; training uses 0-3
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
from pathlib import Path
import json

BASE_DIR = Path(os.environ.get("MA_BASE", "/data/MA_Atul_Ares"))
OUT_DIR  = BASE_DIR / "sept_run" / "comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── datasets ──
# The PURE synthetic export, not the mixed one. Tables A/D below are labelled
# "synthetic test split", and the mixed test split contains 14 real photographs —
# scoring those and calling the result synthetic would be wrong. Real performance is
# measured separately, on TRAB_COCO.
SYN_COCO   = BASE_DIR / "insect_dataset_coco_20000_mix"   # synthetic only, 22 classes
SYN_SPLIT  = "test"

# The MIXED dataset's own splits — what the models were actually trained on and what
# each training notebook reported. Scoring them here does two things the synthetic-only
# table cannot: it cross-checks this notebook's scorer against the numbers Ultralytics
# and RF-DETR printed during training (they should broadly agree; a large gap means one
# of the implementations is measuring something different), and comparing valid against
# test says whether the models are tuned to the split used for early stopping.
MIX_COCO   = BASE_DIR / "insect_dataset_mixed"
# Both flatbug models already have these cached, so scoring RF-DETR on them too is what
# keeps table M a like-for-like comparison rather than a flatbug-only row. It is the
# expensive choice — roughly 3 h per model per split — and `synthetic` covers nearly the
# same ground (mixed_test is the same 5,400 trays plus 14 real images). Kept because a
# partial table is worse than either alternative. Set () to skip.
MIX_SPLITS = ("test", "valid")
TRAB_ZIP   = BASE_DIR / "3DTRAB_Detection_Annotated_v1.zip"
TRAB_DIR   = BASE_DIR / "3DTRAB_Detection_Annotated_v1" # unzipped (images/ + labels/)
# THE HELD-OUT REAL IMAGES ONLY. Roughly half of 3DTRAB is in the training set now,
# so scoring against the full set would be measuring memorisation. This file is
# written by mix_real_and_synthetic.ipynb and lists the 14 test-split photographs;
# 3dtrab_seen_in_training.txt lists the 77 that are off limits.
TRAB_COCO  = BASE_DIR / "insect_dataset_mixed" / "3dtrab_coco_TESTSPLIT.json"
assert TRAB_COCO.exists(), (
    f"{TRAB_COCO} is missing — run mix_real_and_synthetic.ipynb first. Do NOT fall "
    "back to the full 3DTRAB: 77 of its 91 images are in this training set.")

# ── tiling / inference ──
TILE, OVERLAP        = 1536, 256   # matches the tiled training config
MERGE_IOU            = 0.6
EDGE_MARGIN, EDGE_FACTOR = 31, 0.85   # ~2% of a 1536 tile, matching the tiled training
                                      # config. 16 was 2% of an 800 tile, 1% of this one.
CONF_EVAL            = 0.05     # low, so AP sees the full ranking
# Predictions below CONF_EVAL are never emitted, so it is a hard floor on recall —
# and it IS binding here: 60-70% of RF-DETR's boxes land under 0.10, piled against
# the cut-off. This second pass re-runs the real data with the floor dropped, which
# is the only way to actually detect MORE (lowering a *scoring* threshold cannot:
# mAP already integrates every prediction above the floor).
LOW_CONF_PASS        = True
CONF_EVAL_LOW        = 0.005
IOU_NMS              = 0.5
MAX_IMAGES           = None     # None = all; set e.g. 200 for a quick dry run
# The synthetic GT box is the ROTATED CANVAS of the pasted cutout, not the insect
# (generator does rotate(expand=True) then records .size without re-measuring alpha).
# Uniform rotation inflates it by E[|cos|+|sin|] = 4/pi = 1.273 for a round body,
# less for an elongated one; measured 1.23x on 3DTRAB. So predictions run large
# against tightly-annotated real data. 0.80 is geometry-derived, NOT test-fitted.
# Shrink applied to PREDICTED boxes for the *_corr columns. In August this was 0.80:
# the models had been trained on ground truth inflated 1.295x by the rotated-canvas
# bug, so they predicted inflated boxes, and undoing that lifted mAP@50 on real data
# from 0.259 to 0.465. The September dataset has correct boxes, so the models now
# predict correct ones and NO correction is due — leaving 0.80 would shrink good
# predictions by 20% and understate every model.
#
# Kept at 1.0 rather than deleted: the *_corr columns should now equal the raw ones,
# and table D's sweep should peak at 1.0. That peak IS the evidence the box bug is
# gone. If D peaks at 0.8 again, something is still inflating the boxes.
BOX_SCALE            = 1.00

# ── uniform tiled inference: EVERY model gets the identical treatment ──
# Fixed 1:1 tiling is only right when the image has the same pixels-per-insect
# as the training trays. 3DTRAB is 3473 px wide vs the synthetic trays' 1536, so
# at 1:1 a real beetle arrives ~2.9x larger than anything the models saw and gets
# fragmented. Normalising each image to the training tray width first puts every
# model on the same footing AND at the scale it was trained for.
UNIFORM_TILED        = False   # in-memory variant; the downscaled folder below
                               # does the same thing and is inspectable
# Width of the synthetic trays the models trained on. In August this was 1536 while
# real trays were 3473, so table S existed to downscale real data onto the training
# scale. The generator now renders trays at 3473 px = 10.85 px/mm, the same as the
# real ones, so this normalisation is very nearly the identity — table S should now
# agree with table B, and that agreement IS the result: the scale correction that was
# needed in August is not needed any more.
NORM_TRAY_W          = 3473
# Median insect width in those trays, measured from the models' OWN high-confidence
# predictions on the synthetic test split (so it uses no ground truth). Used only
# to REPORT how close the downscale gets to the trained object scale.
TRAIN_OBJ_PX         = 70      # median synthetic insect, sqrt(box area), measured on
                               # the September export. Was 45 when trays were 1536 px.

# ── model registry — one row per TRAINING NOTEBOOK ──────────
#   (label, kind, output_folder, mode, imgsz, source notebook)
#   kind   : "ultralytics" (flatbug) | "rfdetr"
#   mode   : "whole" | "tiled"   <- each model is run the way it was TRAINED
#   folder : the training-output directory; the weights file is RESOLVED inside
#            it below, so a differently-named run subfolder still works.
REGISTRY = [
 ("flatbug whole @1536",  "ultralytics", "sept_run/flatbug_whole_1536",  "whole", 1536, "train_flatbug_fullres.ipynb"),
 ("flatbug tiled @1536",  "ultralytics", "sept_run/flatbug_tiled_1536",  "tiled", 1536, "train_flatbug_tiled_fullres.ipynb"),
 ("RF-DETR whole @1536",  "rfdetr",      "sept_run/rfdetr_whole_1536",   "whole", 1536, "train_rfdetr_fullres.ipynb"),
 ("RF-DETR tiled @1536",  "rfdetr",      "sept_run/rfdetr_tiled_1536",   "tiled", 1536, "train_rfdetr_tiled_fullres.ipynb"),
 # The tiled run continued past its 30-epoch cut-off. Included so the extra training
 # can be read directly against the 30-epoch row — the rare classes are the question.
 ("RF-DETR tiled +cont",  "rfdetr",      "sept_run/rfdetr_tiled_1536_cont", "tiled", 1536, "train_rfdetr_tiled_fullres.ipynb"),
 # two-stage: real flat-bug pretrain then synthetic fine-tune. Skipped
 # automatically if the folder is not on disk, so leaving these in costs nothing.
 ("RF-DETR 2-stage @1536",       "rfdetr", "sept_run/rfdetr_twostage_1536",       "whole", 1536, "train_rfdetr_twostage_fullres.ipynb"),
 ("RF-DETR 2-stage tiled @1536", "rfdetr", "sept_run/rfdetr_twostage_tiled_1536", "tiled", 1536, "train_rfdetr_twostage_tiled_fullres.ipynb"),
]
RF_CKPTS = ["checkpoint_best_total.pth", "checkpoint_best_ema.pth", "checkpoint_best_regular.pth"]

def resolve_weights(kind, folder):
    """Find the final checkpoint inside a training-output folder."""
    d = BASE_DIR / folder
    if not d.is_dir():
        return None, "folder not on disk"
    if kind == "ultralytics":
        cands = sorted(d.glob("*/weights/best.pt")) or sorted(d.glob("**/weights/best.pt"))
    else:
        cands = [d/n for n in RF_CKPTS if (d/n).exists()]
        if not cands:                       # checkpoint one level down
            cands = [q for n in RF_CKPTS for q in sorted(d.glob(f"*/{n}"))]
        # never pick up a stage-1 pretrain by accident
        cands = [q for q in cands if "stage1" not in str(q).lower()]
    if not cands:
        return None, "folder present but no checkpoint inside"
    best = max(cands, key=lambda q: q.stat().st_mtime)   # newest, if several runs
    note = "ok" if len(cands) == 1 else f"ok ({len(cands)} candidates -> newest)"
    return best, note

MODELS, MISSING = [], []
for lab, kind, folder, mode, sz, src in REGISTRY:
    w, note = resolve_weights(kind, folder)
    (MODELS if w else MISSING).append((lab, kind, w, mode, sz) if w else (lab, folder, note))

print(f"{len(MODELS)} of {len(REGISTRY)} model(s) resolved\n")
print(f"   {'model':24s} {'kind':12s} {'mode':6s} {'res':>5s}  weights")
for lab, kind, w, mode, sz in MODELS:
    print(f"   {lab:24s} {kind:12s} {mode:6s} {sz:>5d}  {w.relative_to(BASE_DIR)}")
if MISSING:
    print("\nNOT FOUND (skipped, comparison still runs on the rest):")
    for lab, folder, note in MISSING:
        print(f"   {lab:24s} {folder:36s} {note}")


In [ ]:
# ── Tiling core (grid, box tiling, NMS, edge penalty, merge) ──
import json, math
import numpy as np
from pathlib import Path


def tile_grid(W, H, tile, overlap):
    """Top-left corners covering WxH with `tile` windows and `overlap` px overlap.
    Last row/col is flush to the edge so nothing is missed."""
    step = max(1, tile - overlap)
    xs = list(range(0, max(1, W - tile + 1), step))
    ys = list(range(0, max(1, H - tile + 1), step))
    if not xs or xs[-1] != max(0, W - tile): xs.append(max(0, W - tile))
    if not ys or ys[-1] != max(0, H - tile): ys.append(max(0, H - tile))
    return sorted(set(xs)), sorted(set(ys))


def tile_boxes(boxes, x0, y0, tile, W, H, min_vis=0.85, min_size=4):
    """Clip xyxy boxes into the tile at (x0,y0). Returns (kept_boxes_local, kept_idx).
    A box is kept only if >= min_vis of its area survives the crop, so partial
    insects at tile edges are not learned as whole objects (they are complete in
    the overlapping neighbour tile)."""
    if len(boxes) == 0:
        return np.zeros((0, 4)), []
    b = np.asarray(boxes, float)
    x1 = np.clip(b[:, 0], x0, x0 + tile); y1 = np.clip(b[:, 1], y0, y0 + tile)
    x2 = np.clip(b[:, 2], x0, x0 + tile); y2 = np.clip(b[:, 3], y0, y0 + tile)
    w, h = np.clip(x2 - x1, 0, None), np.clip(y2 - y1, 0, None)
    area_clip = w * h
    area_orig = np.clip((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]), 1e-9, None)
    keep = (area_clip / area_orig >= min_vis) & (w >= min_size) & (h >= min_size)
    idx = np.where(keep)[0]
    out = np.stack([x1[idx] - x0, y1[idx] - y0, x2[idx] - x0, y2[idx] - y0], 1) if len(idx) else np.zeros((0, 4))
    return out, idx.tolist()


def nms(boxes, scores, iou_thr=0.5):
    """Plain greedy NMS. boxes xyxy."""
    if len(boxes) == 0:
        return []
    b = np.asarray(boxes, float); s = np.asarray(scores, float)
    x1, y1, x2, y2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    area = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    order = s.argsort()[::-1]; keep = []
    while order.size:
        i = order[0]; keep.append(int(i))
        if order.size == 1: break
        xx1 = np.maximum(x1[i], x1[order[1:]]); yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]]); yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
        iou = inter / np.clip(area[i] + area[order[1:]] - inter, 1e-9, None)
        order = order[1:][iou <= iou_thr]
    return keep


def edge_penalty(local_boxes, tile, margin=16, factor=0.85):
    """Multiplier that demotes detections touching a tile border. The same object
    is usually complete in the overlapping neighbour tile, so NMS should prefer
    that copy. Mirrors flatbug's EDGE_CASE_MARGIN idea."""
    if len(local_boxes) == 0:
        return np.zeros(0)
    b = np.asarray(local_boxes, float)
    touch = ((b[:, 0] <= margin) | (b[:, 1] <= margin) |
             (b[:, 2] >= tile - margin) | (b[:, 3] >= tile - margin))
    return np.where(touch, factor, 1.0)


def merge_tiles(preds, iou_thr=0.5, per_class=False):
    """preds: list of (boxes_global Nx4, scores N, classes N) from each tile.
    -> merged (boxes, scores, classes) after NMS in full-image coordinates."""
    B = [p[0] for p in preds if len(p[0])]
    if not B:
        return np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)
    boxes = np.concatenate(B)
    scores = np.concatenate([p[1] for p in preds if len(p[0])])
    classes = np.concatenate([p[2] for p in preds if len(p[0])]).astype(int)
    keep_all = []
    for c in (np.unique(classes) if per_class else [None]):
        m = np.where(classes == c)[0] if c is not None else np.arange(len(boxes))
        k = nms(boxes[m], scores[m], iou_thr)
        keep_all += m[k].tolist()
    keep_all = np.array(sorted(keep_all), int)
    return boxes[keep_all], scores[keep_all], classes[keep_all]


## 1. Prepare 3DTRAB — unzip and convert Label Studio JSON to COCO


In [ ]:
# ── Unzip 3DTRAB if needed, then convert Label Studio JSON -> COCO ──
import zipfile, shutil
if not TRAB_DIR.exists():
    assert TRAB_ZIP.exists(), f"missing {TRAB_ZIP}"
    print(f"unzipping {TRAB_ZIP} ...")
    with zipfile.ZipFile(TRAB_ZIP) as z: z.extractall(BASE_DIR)
TRAB_IMGS   = TRAB_DIR / "images"
TRAB_LABELS = TRAB_DIR / "labels" / "labels.json"
assert TRAB_IMGS.is_dir() and TRAB_LABELS.exists(), f"unexpected layout under {TRAB_DIR}"

import json, urllib.parse
from collections import Counter

CLASS_NAMES = [
    # 22 classes. Siphonaptera and Raphidioptera dropped (6 and 2 specimens);
    # Diplopoda -> Polyxenida, Chilopoda -> Lithobiomorpha + Scutigeromorpha.
    "Lepidoptera",
    "Psocodea",
    "Odonata",
    "Zoraptera",
    "Phasmatodea",
    "Zygentoma",
    "Thysanoptera",
    "Strepsiptera",
    "Archaeognatha",
    "Dermaptera",
    "Diptera",
    "Coleoptera",
    "Hymenoptera",
    "Hemiptera",
    "Orthoptera",
    "Trichoptera",
    "Neuroptera",
    "Araneae",
    "Opiliones",
    "Polyxenida",
    "Lithobiomorpha",
    "Scutigeromorpha",
]
# annotator vocabulary -> our 22-class taxonomy
NAME_MAP = {"Coleoptera":"Coleoptera", "Orthoptera":"Orthoptera", "Diptera":"Diptera",
            "Aranea":"Araneae",            # spelling
            "Heteroptera":"Hemiptera"}     # Heteroptera is a suborder of Hemiptera

def convert(labels_json, out_json=None):
    d = json.load(open(labels_json))
    name2id = {n: i+1 for i, n in enumerate(CLASS_NAMES)}
    images, anns, aid = [], [], 1
    unmapped, unlabelled = Counter(), 0
    for iid, t in enumerate(d, 1):
        fn = urllib.parse.unquote(t["data"]["image"]).replace("\\", "/").split("/")[-1]
        res = (t.get("annotations") or [{}])[0].get("result", [])
        ch = {r["id"]: (r["value"].get("choices") or [None])[0]
              for r in res if r.get("type") == "choices"}
        W = H = None
        for r in res:
            if r.get("type") != "polygonlabels":
                continue
            W, H = r["original_width"], r["original_height"]
            pts = r["value"]["points"]                       # PERCENT of W,H
            xs = [p[0]/100.0*W for p in pts]; ys = [p[1]/100.0*H for p in pts]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
            x1, y1 = max(0.0, x1), max(0.0, y1)
            x2, y2 = min(float(W), x2), min(float(H), y2)
            if x2 - x1 < 1 or y2 - y1 < 1:
                continue
            raw = ch.get(r["id"])
            if raw is None:
                unlabelled += 1; continue                    # no order label -> skip for class eval
            mapped = NAME_MAP.get(raw)
            if mapped is None:
                unmapped[raw] += 1; continue
            anns.append({"id": aid, "image_id": iid, "category_id": name2id[mapped],
                         "bbox": [x1, y1, x2-x1, y2-y1], "area": (x2-x1)*(y2-y1),
                         "iscrowd": 0, "raw_label": raw}); aid += 1
        if W:
            images.append({"id": iid, "file_name": fn, "width": W, "height": H})
    coco = {"info": {"description": "3DTRAB_Detection_Annotated_v1 (polygon-derived boxes)"},
            "categories": [{"id": i+1, "name": n, "supercategory": "none"}
                           for i, n in enumerate(CLASS_NAMES)],
            "images": images, "annotations": anns}
    if out_json: json.dump(coco, open(out_json, "w"))
    return coco, unlabelled, unmapped


# LOAD the held-out file; do NOT regenerate it.
#
# TRAB_COCO is 3dtrab_coco_TESTSPLIT.json, written by mix_real_and_synthetic.ipynb: the
# 14 real photographs whose tray groups were kept out of training. Calling
# convert(TRAB_LABELS, TRAB_COCO) here would OVERWRITE it with all 91 images — 77 of
# which the models trained on — and every real-data number below would be measuring
# memorisation. The converter is still defined above because the annotation-parsing
# logic is worth keeping in one place, but it is not run against TRAB_COCO.
assert TRAB_COCO.exists(), (
    f"{TRAB_COCO} not found. It is written by mix_real_and_synthetic.ipynb. Do NOT "
    f"substitute the full 3DTRAB set — 77 of its 91 images are in the training data.")
coco = json.load(open(TRAB_COCO))
_seen_f = TRAB_COCO.parent / "3dtrab_seen_in_training.txt"
_seen = set(_seen_f.read_text().split()) if _seen_f.exists() else set()
_leak = sorted({i["file_name"] for i in coco["images"]} & _seen)
assert not _leak, (
    f"{len(_leak)} images in the evaluation set were also in training: {_leak[:5]}\n"
    "This file is not the held-out split.")
print(f"3DTRAB held-out evaluation set: {len(coco['images'])} images, "
      f"{len(coco['annotations'])} boxes")
print(f"  ({len(_seen)} further images are in the training set and are excluded)"
      if _seen else "  (3dtrab_seen_in_training.txt not found — cannot cross-check)")
_unl, _unm = 0, {}

# --- verification ---
from collections import Counter
import numpy as np
_n2 = {c["id"]: c["name"] for c in coco["categories"]}
_dist = Counter(_n2[a["category_id"]] for a in coco["annotations"])
_wh = {i["id"]: (i["width"], i["height"]) for i in coco["images"]}
_bad = sum(1 for a in coco["annotations"] if a["bbox"][0] < 0 or a["bbox"][1] < 0
           or a["bbox"][0]+a["bbox"][2] > _wh[a["image_id"]][0]+.01
           or a["bbox"][1]+a["bbox"][3] > _wh[a["image_id"]][1]+.01)
_fr = np.array([[a["bbox"][2]/_wh[a["image_id"]][0], a["bbox"][3]/_wh[a["image_id"]][1]]
                for a in coco["annotations"]])
print(f"  classes present: {dict(_dist)}")
print(f"  out-of-bounds boxes: {_bad} | median box {np.median(_fr[:,0])*100:.2f}% x {np.median(_fr[:,1])*100:.2f}%")
assert _bad == 0 and np.median(_fr) < 0.25, "3DTRAB boxes look wrong - STOP"
_missing_files = [i["file_name"] for i in coco["images"] if not (TRAB_IMGS/i["file_name"]).exists()]
print(f"  images missing on disk: {len(_missing_files)}" + (f" e.g. {_missing_files[:2]}" if _missing_files else ""))
assert not _missing_files, "image filenames do not match the images/ folder"
TRAB_PRESENT = sorted(_dist)          # the 5 classes actually annotated
print(f"\nOK. 3DTRAB annotated classes: {TRAB_PRESENT}")


## 1b. Downscale 3DTRAB to the training tray scale  *(new folder; originals untouched)*


In [ ]:
# ── Build a DOWNSCALED copy of 3DTRAB at the training tray scale ──
# Kept from August, when synthetic trays were 1536 px and real ones 3473, so a real
# beetle arrived ~2.9x larger than anything the model had seen. The generator now
# renders at 3473 px, matching real, so this is close to a no-op (it only removes the
# few-pixel variation between real images, 3457-3481 px). Retained deliberately: if
# table S still agrees with table B, that is direct evidence the scale gap is closed.
# The originals are never touched; annotations are scaled by the same factor, and IoU
# is invariant under uniform scaling.
from PIL import Image
import numpy as np

TRAB_DS_DIR  = OUT_DIR / f"3dtrab_downscaled_{NORM_TRAY_W}"
TRAB_DS_IMGS = TRAB_DS_DIR / "images"
TRAB_DS_COCO = OUT_DIR / f"3dtrab_coco_{NORM_TRAY_W}.json"
TRAB_DS_IMGS.mkdir(parents=True, exist_ok=True)

_src = json.load(open(TRAB_COCO))
_scale, _made, _kept = {}, 0, 0
for im in _src["images"]:
    s = NORM_TRAY_W / float(im["width"])
    _scale[im["id"]] = s
    dst = TRAB_DS_IMGS / im["file_name"]
    if dst.exists():
        _kept += 1
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(TRAB_IMGS / im["file_name"]) as _i:
        _i.convert("RGB").resize(
            (max(1, round(im["width"] * s)), max(1, round(im["height"] * s))),
            Image.LANCZOS).save(dst, quality=95)
    _made += 1

_ds = {"info": {"description": f"3DTRAB downscaled to {NORM_TRAY_W}px tray width"},
       "categories": _src["categories"], "images": [], "annotations": []}
for im in _src["images"]:
    s = _scale[im["id"]]
    _ds["images"].append({**im, "width":  round(im["width"] * s),
                                "height": round(im["height"] * s)})
for a in _src["annotations"]:
    s = _scale[a["image_id"]]
    x, y, w, h = a["bbox"]
    _ds["annotations"].append({**a, "bbox": [x*s, y*s, w*s, h*s],
                               "area": float(w*s * h*s)})
json.dump(_ds, open(TRAB_DS_COCO, "w"))

# --- verification: boxes must land at the SAME relative size as before ---
_wh0 = {i["id"]: (i["width"], i["height"]) for i in _src["images"]}
_wh1 = {i["id"]: (i["width"], i["height"]) for i in _ds["images"]}
_f0 = np.median([a["bbox"][2] / _wh0[a["image_id"]][0] for a in _src["annotations"]])
_f1 = np.median([a["bbox"][2] / _wh1[a["image_id"]][0] for a in _ds["annotations"]])
_bad = sum(1 for a in _ds["annotations"]
           if a["bbox"][0] < -0.01 or a["bbox"][1] < -0.01
           or a["bbox"][0] + a["bbox"][2] > _wh1[a["image_id"]][0] + 1
           or a["bbox"][1] + a["bbox"][3] > _wh1[a["image_id"]][1] + 1)
_px0 = np.median([a["bbox"][2] for a in _src["annotations"]])
_px1 = np.median([a["bbox"][2] for a in _ds["annotations"]])

print(f"downscaled 3DTRAB -> {TRAB_DS_IMGS}")
print(f"  {_made} images written, {_kept} already present "
      f"(delete the folder to rebuild)")
print(f"  scale factor: {min(_scale.values()):.3f}–{max(_scale.values()):.3f}")
print(f"  median box: {_px0:.0f} px -> {_px1:.0f} px "
      f"(models trained on ~{TRAIN_OBJ_PX} px)")
print(f"  object-scale mismatch: {_px0/TRAIN_OBJ_PX:.2f}x -> {_px1/TRAIN_OBJ_PX:.2f}x")
if _px1/TRAIN_OBJ_PX > 1.5:
    print(f"    NOTE: still {_px1/TRAIN_OBJ_PX:.2f}x large. Normalising by TRAY WIDTH is")
    print(f"    GT-free and defensible, but 3DTRAB insects are relatively bigger than")
    print(f"    the synthetic ones ({_f1*100:.2f}% of image width vs "
          f"{TRAIN_OBJ_PX/NORM_TRAY_W*100:.2f}%), so a residual remains.")
print(f"  box size as a fraction of image: {_f0*100:.2f}% -> {_f1*100:.2f}% "
      f"(must be unchanged)")
print(f"  out-of-bounds boxes: {_bad}")
assert _bad == 0, "downscaled boxes fall outside the image - STOP"
assert abs(_f0 - _f1) < 0.002, "relative box size changed - the scaling is wrong"
_miss = [i["file_name"] for i in _ds["images"]
         if not (TRAB_DS_IMGS / i["file_name"]).exists()]
assert not _miss, f"missing downscaled images: {_miss[:3]}"
print(f"\nOK. Originals untouched at {TRAB_IMGS}")


## 2. Predictor and metrics


In [ ]:
# ── Unified predictor: {ultralytics, rfdetr} x {whole, tiled} ──
import sys, time
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

_src = str(BASE_DIR / "rf-detr-1.7.1" / "src")
if _src not in sys.path: sys.path.insert(0, _src)

def safe(label):
    """filename-safe model label (must match between the run and score cells)"""
    return label.replace(" ", "_").replace("@", "")

def pred_path(label, tag):
    return OUT_DIR / f"preds__{tag}__{safe(label)}.json"

_CACHE = {}
def load_model(kind, weights, imgsz):
    key = (kind, str(weights), imgsz)
    if key in _CACHE: return _CACHE[key]
    if kind == "ultralytics":
        # Bind torch._utils explicitly. Unpickling a .pt resolves
        # torch._utils._rebuild_tensor_v2, and on torch 2.13 (the flatbug env)
        # that submodule is not always bound by a bare `import torch`, which
        # surfaces as "module 'torch' has no attribute '_utils'".
        import torch, torch._utils  # noqa: F401
        from ultralytics import YOLO
        m = YOLO(str(weights))
    else:
        from rfdetr import RFDETR
        # from_checkpoint does NOT restore resolution -> pass it explicitly
        m = RFDETR.from_checkpoint(str(weights), resolution=imgsz)
    _CACHE[key] = m
    return m

def _predict_crop(m, kind, img, imgsz, name_to_idx, conf=None):
    conf = CONF_EVAL if conf is None else conf
    """-> boxes(N,4) xyxy in CROP coords, scores(N), classes(N)"""
    if kind == "ultralytics":
        r = m.predict(img, conf=conf, iou=IOU_NMS, imgsz=imgsz, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0:
            return np.zeros((0,4)), np.zeros(0), np.zeros(0,int)
        return (r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(),
                r.boxes.cls.cpu().numpy().astype(int))
    d = m.predict(img, threshold=conf)
    if d is None or len(d) == 0:
        return np.zeros((0,4)), np.zeros(0), np.zeros(0,int)
    b = np.asarray(d.xyxy, float).reshape(-1,4); s = np.asarray(d.confidence, float)
    nm = d.data.get("class_name", None)
    if nm is not None and len(nm) == len(s):
        c = np.array([name_to_idx.get(str(x), -1) for x in nm])
    else:
        c = np.asarray(d.class_id, int)
        if c.size and c.max() >= len(name_to_idx): c = c - 1      # 1-indexed COCO ids
    k = c >= 0
    return b[k], s[k], c[k]

def predict_image(m, kind, mode, imgsz, path, name_to_idx, conf=None):
    """mode: 'whole' | 'tiled' (1:1) | 'tiled_norm' (scale-normalised, uniform)"""
    img = Image.open(path).convert("RGB"); W, H = img.size
    if mode == "whole":
        return _predict_crop(m, kind, img, imgsz, name_to_idx, conf)
    nrm = 1.0                       # NOT `s` — that name is the score array below
    if mode == "tiled_norm" and NORM_TRAY_W and W != NORM_TRAY_W:
        nrm = NORM_TRAY_W / float(W)                     # tray -> training width
        img = img.resize((max(1, round(W*nrm)), max(1, round(H*nrm))), Image.BILINEAR)
        W, H = img.size
    tp = []
    xs, ys = tile_grid(W, H, TILE, OVERLAP)
    for y0 in ys:
        for x0 in xs:
            crop = img.crop((x0, y0, min(x0+TILE, W), min(y0+TILE, H)))
            b, s, c = _predict_crop(m, kind, crop, imgsz, name_to_idx, conf)
            if not len(b): continue
            s = s * edge_penalty(b, TILE, EDGE_MARGIN, EDGE_FACTOR)
            tp.append((b + np.array([x0, y0, x0, y0]), s, c))
    # CLASS-AGNOSTIC across tiles. With per_class=True the same physical insect
        # detected as Coleoptera in one tile and Diptera in the overlapping one
        # survived as TWO boxes, because NMS never compared them. Measured on the
        # August predictions: 31% of RF-DETR tiled's detections were co-located
        # pairs with different labels (RF-DETR whole: 5%, flatbug: 0%). That alone
        # explains the tiled models' collapse on real data — 55,883 predictions for
        # 3,072 ground truths. Agnostic NMS keeps the highest-scoring label.
        b, sc, c = merge_tiles(tp, iou_thr=MERGE_IOU, per_class=False)
    if nrm != 1.0 and len(b):
        b = np.asarray(b, float) / nrm                   # back to ORIGINAL coords
    return b, sc, c

def run_dataset(label, kind, weights, mode, imgsz, coco_path, img_dir, tag, conf=None):
    """Run one model over one dataset, cache predictions to disk."""
    out_f = pred_path(label, tag)
    if out_f.exists():
        print(f"   [cached] {out_f.name}"); return out_f
    ann = json.load(open(coco_path))
    cats = sorted(ann["categories"], key=lambda c: c["id"])
    name_to_idx = {c["name"]: i for i, c in enumerate(cats)}
    metas = ann["images"] if MAX_IMAGES is None else ann["images"][:MAX_IMAGES]
    m = load_model(kind, weights, imgsz)
    out, t0 = {}, time.time()
    for im in tqdm(metas, desc=f"{label} | {tag}", leave=False):
        p = Path(img_dir)/im["file_name"]
        if not p.exists(): continue
        b, s, c = predict_image(m, kind, mode, imgsz, p, name_to_idx, conf)
        out[str(im["id"])] = {"boxes": np.asarray(b).reshape(-1,4).tolist(),
                              "scores": np.asarray(s).tolist(),
                              "classes": np.asarray(c, int).tolist()}
    json.dump({"model": label, "mode": mode, "imgsz": imgsz, "kind": kind,
               "conf": CONF_EVAL if conf is None else conf,
               "seconds": time.time()-t0, "images": out}, open(out_f, "w"))
    print(f"   {label:24s} {sum(len(v['scores']) for v in out.values()):7d} dets "
          f"over {len(out)} images in {time.time()-t0:.0f}s")
    return out_f
print("predictor ready")


In [ ]:
# ── Metrics: COCO 101-point AP over IoU 0.50:0.95, greedy class-aware matching ──
import numpy as np, pandas as pd
IOU_VEC = np.linspace(0.5, 0.95, 10)

def iou_mat(a, b):
    if len(a)==0 or len(b)==0: return np.zeros((len(a), len(b)))
    a, b = np.asarray(a,float), np.asarray(b,float)
    lt = np.maximum(a[:,None,:2], b[None,:,:2]); rb = np.minimum(a[:,None,2:], b[None,:,2:])
    wh = np.clip(rb-lt, 0, None); inter = wh[...,0]*wh[...,1]
    aa = (a[:,2]-a[:,0])*(a[:,3]-a[:,1]); ab = (b[:,2]-b[:,0])*(b[:,3]-b[:,1])
    return inter/np.clip(aa[:,None]+ab[None]-inter, 1e-9, None)

# NumPy 2.0 removed np.trapz in favour of np.trapezoid. The training envs run
# different NumPy majors (flatbug torch 2.13 / rfdetr torch 2.9), so bind once
# and use _TRAPZ rather than depending on which one is installed.
_TRAPZ = getattr(np, "trapezoid", None) or np.trapz
def _ap(rec, prec):
    mrec = np.concatenate(([0.],rec,[1.])); mpre = np.concatenate(([1.],prec,[0.]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0,1,101); return float(_TRAPZ(np.interp(x,mrec,mpre),x))

def _agnostic_nms(b, s, thr=0.5):
    """Class-agnostic NMS. Used for the SINGLE-CLASS view: a model that puts two
    differently-labelled boxes on one insect has made a CLASSIFICATION error, and
    should not also be charged a false positive in a detection-only metric."""
    if len(b) == 0: return np.zeros(0, int)
    b = np.asarray(b, float); s = np.asarray(s, float)
    x1,y1,x2,y2 = b[:,0],b[:,1],b[:,2],b[:,3]
    ar = np.clip(x2-x1,0,None)*np.clip(y2-y1,0,None)
    order = s.argsort()[::-1]; keep = []
    while order.size:
        i = order[0]; keep.append(int(i))
        if order.size == 1: break
        xx1=np.maximum(x1[i],x1[order[1:]]); yy1=np.maximum(y1[i],y1[order[1:]])
        xx2=np.minimum(x2[i],x2[order[1:]]); yy2=np.minimum(y2[i],y2[order[1:]])
        inter=np.clip(xx2-xx1,0,None)*np.clip(yy2-yy1,0,None)
        iou=inter/np.clip(ar[i]+ar[order[1:]]-inter,1e-9,None)
        order = order[1:][iou <= thr]
    return np.array(keep, int)

def _scale_boxes(P, k):
    """Shrink/grow every predicted box about its own centre by factor k."""
    if k == 1.0: return P
    P = {**P, "images": {i: dict(v) for i, v in P["images"].items()}}
    for v in P["images"].values():
        b = np.array(v["boxes"], float).reshape(-1, 4)
        if not len(b): continue
        cx, cy = (b[:,0]+b[:,2])/2, (b[:,1]+b[:,3])/2
        w, h = (b[:,2]-b[:,0])*k/2, (b[:,3]-b[:,1])*k/2
        v["boxes"] = np.stack([cx-w, cy-h, cx+w, cy+h], 1).tolist()
    return P

def evaluate(pred_file, coco_path, single_class=False, restrict_to=None, agnostic_nms=True,
             box_scale=1.0):
    """restrict_to: list of class NAMES to average over (None = all with GT).
    agnostic_nms: only used when single_class=True.
    box_scale:  resize predicted boxes about their centre before matching. The
                synthetic labels are the rotated-canvas box (see the diagnostic
                cell), so predictions run ~1.25x large against tight annotations;
                0.80 is the factor the rotation geometry predicts."""
    P = json.load(open(pred_file)); ann = json.load(open(coco_path))
    P = _scale_boxes(P, box_scale)
    cats = sorted(ann["categories"], key=lambda c: c["id"])
    names = [c["name"] for c in cats]; c2i = {c["id"]: i for i,c in enumerate(cats)}
    NC = 1 if single_class else len(names)
    GT = {}
    for a in ann["annotations"]:
        b = a["bbox"]
        GT.setdefault(a["image_id"], []).append(
            (0 if single_class else c2i[a["category_id"]], b[0],b[1],b[0]+b[2],b[1]+b[3]))
    tps, cf, pc, tc = [], [], [], []
    for iid_s, pr in P["images"].items():
        iid = int(iid_s); g = GT.get(iid, [])
        gb = np.array([x[1:] for x in g]).reshape(-1,4); gc = np.array([x[0] for x in g], int)
        pb = np.array(pr["boxes"]).reshape(-1,4); ps = np.array(pr["scores"])
        pcl = np.zeros(len(ps), int) if single_class else np.array(pr["classes"], int)
        if single_class and agnostic_nms and len(pb):
            _k = _agnostic_nms(pb, ps, IOU_NMS)
            pb, ps, pcl = pb[_k], ps[_k], pcl[_k]
        tp = np.zeros((len(pb), len(IOU_VEC)), bool)
        if len(pb) and len(gb):
            M = iou_mat(pb, gb); same = pcl[:,None]==gc[None,:]
            for ti,thr in enumerate(IOU_VEC):
                ci,gi = np.where((M>=thr)&same)
                if not len(ci): continue
                o = np.argsort(-M[ci,gi]); up,ug = set(),set()
                for c_,g_ in zip(ci[o],gi[o]):
                    if c_ in up or g_ in ug: continue
                    up.add(c_); ug.add(g_); tp[c_,ti]=True
        tps.append(tp); cf.append(ps); pc.append(pcl); tc.append(gc)
    tp = np.concatenate(tps) if tps else np.zeros((0,len(IOU_VEC)),bool)
    conf = np.concatenate(cf) if cf else np.zeros(0)
    pcls = np.concatenate(pc) if pc else np.zeros(0,int)
    tcls = np.concatenate(tc) if tc else np.zeros(0,int)
    o = np.argsort(-conf); tp,conf,pcls = tp[o],conf[o],pcls[o]
    n_gt = np.bincount(tcls, minlength=NC)
    keep = set(range(NC)) if restrict_to is None else {names.index(n) for n in restrict_to}
    rows, A50, A75, A5095, Ps, Rs = [], [], [], [], [], []
    for c in range(NC):
        if n_gt[c]==0 or c not in keep: continue
        i = pcls==c
        if i.sum()==0:
            rows.append({"class":"insect" if single_class else names[c],"instances":int(n_gt[c]),
                         "AP50":0.,"AP75":0.,"AP50_95":0.})
            A50.append(0.);A75.append(0.);A5095.append(0.);Ps.append(0.);Rs.append(0.); continue
        tpc = tp[i].cumsum(0); fpc = (1-tp[i]).cumsum(0)
        rec = tpc/(n_gt[c]+1e-16); prec = tpc/(tpc+fpc+1e-16)
        aps = [_ap(rec[:,t],prec[:,t]) for t in range(len(IOU_VEC))]
        f1 = 2*prec[:,0]*rec[:,0]/(prec[:,0]+rec[:,0]+1e-16); bi = int(np.argmax(f1))
        A50.append(aps[0]); A75.append(aps[5]); A5095.append(float(np.mean(aps)))
        Ps.append(float(prec[bi,0])); Rs.append(float(rec[bi,0]))
        rows.append({"class":"insect" if single_class else names[c],"instances":int(n_gt[c]),
                     "AP50":round(aps[0],4),"AP75":round(aps[5],4),
                     "AP50_95":round(float(np.mean(aps)),4)})
    return (dict(model=P["model"], mode=P["mode"],
                 mAP50_95=float(np.mean(A5095)) if A5095 else 0.,
                 mAP50=float(np.mean(A50)) if A50 else 0.,
                 mAP75=float(np.mean(A75)) if A75 else 0.,
                 precision=float(np.mean(Ps)) if Ps else 0.,
                 recall=float(np.mean(Rs)) if Rs else 0.,
                 n_pred=int(len(conf)), n_gt=int(sum(n_gt[c] for c in range(NC) if c in keep)),
                 seconds=round(P.get("seconds",0),1)),
            pd.DataFrame(rows))
print("metrics ready")


## 3a. Run flatbug models  *(kernel: `Python (flatbug)`)*


In [ ]:
# [KERNEL: flatbug]
# ============================================================
# RUN A — flatbug models        [KERNEL: Python (flatbug)]
# ============================================================
# Ultralytics checkpoints trained in the flatbug env (torch 2.13). Load them with
# the SAME interpreter that wrote them; torch is picky across major versions.
SYN_ANN = SYN_COCO/SYN_SPLIT/"_annotations.coco.json"
assert SYN_ANN.exists(), f"missing {SYN_ANN}"

_todo = [m for m in MODELS if m[1] == "ultralytics"]
print(f"flatbug models to run: {[m[0] for m in _todo]}\n")
for lab, kind, w, mode, sz in _todo:
    print(f"=== {lab} ({mode} @{sz}) ===")
    run_dataset(lab, kind, w, mode, sz, SYN_ANN,  SYN_COCO/SYN_SPLIT, "synthetic")
    run_dataset(lab, kind, w, mode, sz, TRAB_COCO, TRAB_IMGS,         "3dtrab")
    # downscaled 3DTRAB: native mode, but now at the object scale it was trained on
    run_dataset(lab, kind, w, mode, sz, TRAB_DS_COCO, TRAB_DS_IMGS, "3dtrab_ds")
    # the mixed dataset's own splits — as trained
    for _msp in MIX_SPLITS:
        _mann = MIX_COCO/_msp/"_annotations.coco.json"
        if _mann.exists():
            run_dataset(lab, kind, w, mode, sz, _mann, MIX_COCO/_msp, f"mixed_{_msp}")
        else:
            print(f"    (no {_mann} - skipping mixed_{_msp})")
    if LOW_CONF_PASS:      # same again with the confidence floor dropped
        print(f"    low-confidence pass (floor {CONF_EVAL_LOW})")
        run_dataset(lab, kind, w, mode, sz, TRAB_DS_COCO, TRAB_DS_IMGS,
                    "3dtrab_ds_low", conf=CONF_EVAL_LOW)
    if UNIFORM_TILED:                       # identical inference for every model
        print(f"    uniform tiled (normalised to {NORM_TRAY_W} px)")
        run_dataset(lab, kind, w, "tiled_norm", sz, SYN_ANN,  SYN_COCO/SYN_SPLIT, "synthetic_uni")
        run_dataset(lab, kind, w, "tiled_norm", sz, TRAB_COCO, TRAB_IMGS,         "3dtrab_uni")
print("\nflatbug side done. Now switch kernel to rfdetr-env, re-run CONFIG..METRICS, then RUN B.")


## 3b. Run RF-DETR models  *(switch kernel to `rfdetr-env` first)*


In [ ]:
# [KERNEL: rfdetr]
# ============================================================
# RUN B — RF-DETR models        [KERNEL: rfdetr-env]
# ============================================================
# Switch the kernel, then re-run: CONFIG -> tiling core -> 3DTRAB prep ->
# predictor -> metrics -> this cell. Predictions already on disk are reused.
SYN_ANN = SYN_COCO/SYN_SPLIT/"_annotations.coco.json"
assert SYN_ANN.exists(), f"missing {SYN_ANN}"

_todo = [m for m in MODELS if m[1] == "rfdetr"]
print(f"RF-DETR models to run: {[m[0] for m in _todo]}\n")
for lab, kind, w, mode, sz in _todo:
    print(f"=== {lab} ({mode} @{sz}) ===")
    run_dataset(lab, kind, w, mode, sz, SYN_ANN,  SYN_COCO/SYN_SPLIT, "synthetic")
    run_dataset(lab, kind, w, mode, sz, TRAB_COCO, TRAB_IMGS,         "3dtrab")
    # downscaled 3DTRAB: native mode, but now at the object scale it was trained on
    run_dataset(lab, kind, w, mode, sz, TRAB_DS_COCO, TRAB_DS_IMGS, "3dtrab_ds")
    # the mixed dataset's own splits — as trained
    for _msp in MIX_SPLITS:
        _mann = MIX_COCO/_msp/"_annotations.coco.json"
        if _mann.exists():
            run_dataset(lab, kind, w, mode, sz, _mann, MIX_COCO/_msp, f"mixed_{_msp}")
        else:
            print(f"    (no {_mann} - skipping mixed_{_msp})")
    if LOW_CONF_PASS:      # same again with the confidence floor dropped
        print(f"    low-confidence pass (floor {CONF_EVAL_LOW})")
        run_dataset(lab, kind, w, mode, sz, TRAB_DS_COCO, TRAB_DS_IMGS,
                    "3dtrab_ds_low", conf=CONF_EVAL_LOW)
    if UNIFORM_TILED:                       # identical inference for every model
        print(f"    uniform tiled (normalised to {NORM_TRAY_W} px)")
        run_dataset(lab, kind, w, "tiled_norm", sz, SYN_ANN,  SYN_COCO/SYN_SPLIT, "synthetic_uni")
        run_dataset(lab, kind, w, "tiled_norm", sz, TRAB_COCO, TRAB_IMGS,         "3dtrab_uni")
print("\nRF-DETR side done. Now run the SCORE cell (either kernel).")


## 4. Results


In [ ]:
# ============================================================
# SCORE — reads the cached prediction files from disk   [either kernel]
# ============================================================
import pandas as pd
SYN_ANN = SYN_COCO/SYN_SPLIT/"_annotations.coco.json"

_have, _missing = [], []
for m in MODELS:
    lab = m[0]
    if pred_path(lab,"synthetic").exists() and pred_path(lab,"3dtrab").exists():
        _have.append(m)
    else:
        _missing.append((lab,
            "synthetic" if not pred_path(lab,"synthetic").exists() else "",
            "3dtrab"    if not pred_path(lab,"3dtrab").exists()    else ""))
print(f"scoring {len(_have)} model(s): {[m[0] for m in _have]}")
if _missing:
    print("INCOMPLETE (run the matching RUN cell in the right kernel):")
    for lab, a, b in _missing: print(f"   {lab:24s} missing: {' '.join(x for x in (a,b) if x)}")
assert _have, "no complete prediction sets found"

rowsA, rowsB, rowsC, per_class = [], [], [], {}
rowsM = {sp: [] for sp in MIX_SPLITS}
for lab, kind, w, mode, sz in _have:
    s, dfA = evaluate(pred_path(lab,"synthetic"), SYN_ANN)
    rowsA.append(s); per_class[(lab,"synthetic")] = dfA
    b, _   = evaluate(pred_path(lab,"3dtrab"), TRAB_COCO, single_class=True)
    bc, _  = evaluate(pred_path(lab,"3dtrab"), TRAB_COCO, single_class=True, box_scale=BOX_SCALE)
    b["mAP50_95_corr"] = round(bc["mAP50_95"], 4); b["mAP50_corr"] = round(bc["mAP50"], 4)
    rowsB.append(b)
    c, dfC = evaluate(pred_path(lab,"3dtrab"), TRAB_COCO, restrict_to=TRAB_PRESENT)
    cc, _  = evaluate(pred_path(lab,"3dtrab"), TRAB_COCO, restrict_to=TRAB_PRESENT, box_scale=BOX_SCALE)
    c["mAP50_95_corr"] = round(cc["mAP50_95"], 4); c["mAP50_corr"] = round(cc["mAP50"], 4)
    rowsC.append(c); per_class[(lab,"3dtrab")] = dfC

    # the mixed dataset's own splits, scored with THIS notebook's scorer
    for _msp in MIX_SPLITS:
        _mp = pred_path(lab, f"mixed_{_msp}")
        _ma = MIX_COCO/_msp/"_annotations.coco.json"
        if _mp.exists() and _ma.exists():
            _m, _dfm = evaluate(_mp, _ma)
            rowsM[_msp].append(_m); per_class[(lab, f"mixed_{_msp}")] = _dfm
BASE_COLS = ["model","mode","mAP50_95","mAP50","mAP75","precision","recall","n_pred","n_gt","seconds"]
def show(rows, title, fn):
    extra = [c for c in ("mAP50_95_corr","mAP50_corr") if c in rows[0]]
    cols = BASE_COLS[:5] + extra + BASE_COLS[5:]
    df = pd.DataFrame(rows)[cols]
    df = df.round(4).sort_values("mAP50_95", ascending=False)
    df.to_csv(OUT_DIR/fn, index=False)
    print(f"\n{'='*96}\n{title}\n{'='*96}")
    print(df.to_string(index=False))
    return df
# ── downscaled 3DTRAB: native inference at the trained object scale ──
rowsS = []
for lab, kind, w, mode, sz in _have:
    f = pred_path(lab, "3dtrab_ds")
    if not f.exists(): continue
    b,_  = evaluate(f, TRAB_DS_COCO, single_class=True)
    bc,_ = evaluate(f, TRAB_DS_COCO, single_class=True, box_scale=BOX_SCALE)
    b["mAP50_95_corr"] = round(bc["mAP50_95"],4); b["mAP50_corr"] = round(bc["mAP50"],4)
    rowsS.append(b)

# ── uniform tiled: same inference for every model, scale-normalised ──
rowsUA, rowsUB = [], []
for lab, kind, w, mode, sz in _have:
    fu_s, fu_t = pred_path(lab,"synthetic_uni"), pred_path(lab,"3dtrab_uni")
    if fu_s.exists():
        s,_ = evaluate(fu_s, SYN_ANN); s["mode"] = "uniform"; rowsUA.append(s)
    if fu_t.exists():
        b,_  = evaluate(fu_t, TRAB_COCO, single_class=True)
        bc,_ = evaluate(fu_t, TRAB_COCO, single_class=True, box_scale=BOX_SCALE)
        b["mode"] = "uniform"
        b["mAP50_95_corr"] = round(bc["mAP50_95"],4); b["mAP50_corr"] = round(bc["mAP50"],4)
        rowsUB.append(b)

dfA = show(rowsA, "A. SYNTHETIC test split — 22 classes (held-out, specimen-disjoint)", "A_synthetic_22class.csv")

# ── M. the mixed dataset's own splits — "as trained" ──
for _msp in MIX_SPLITS:
    if not rowsM.get(_msp):
        print(f"\n(no predictions for mixed_{_msp} — run the scoring cells above)")
        continue
    _dfM = show(rowsM[_msp], f"M. MIXED dataset, {_msp.upper()} split — 22 classes, as trained "
                             f"(synthetic + the real images assigned to this split)",
                f"M_mixed_{_msp}.csv")
if all(rowsM.get(sp) for sp in MIX_SPLITS) and len(MIX_SPLITS) == 2:
    _a, _b = MIX_SPLITS
    _da = {r["model"]: r["mAP50_95"] for r in rowsM[_a]}
    _db = {r["model"]: r["mAP50_95"] for r in rowsM[_b]}
    print(f"\n  {_a} vs {_b} (mAP@50:95). A model far better on the split used for early")
    print(f"  stopping than on the other is tuned to it rather than generalising.\n")
    print(f"    {'model':28s}{_a:>10s}{_b:>10s}{'diff':>9s}")
    for k in _da:
        if k in _db:
            print(f"    {k:28s}{_da[k]:>10.4f}{_db[k]:>10.4f}{_da[k]-_db[k]:>+9.4f}")
    print("\n  These are also the numbers the training notebooks printed, so they cross-check")
    print("  this notebook's scorer against Ultralytics' and RF-DETR's. Broad agreement is")
    print("  expected; a large gap means the implementations measure different things.")

dfB = show(rowsB, "B. 3DTRAB real trays, HELD-OUT test split only — SINGLE CLASS (detection + localisation)", "B_3dtrab_singleclass.csv")
dfC = show(rowsC, "C. 3DTRAB real trays, HELD-OUT test split only — the 5 annotated orders", "C_3dtrab_5class.csv")

dfS = None
if rowsS:
    dfS = show(rowsS, f"S. 3DTRAB DOWNSCALED to {NORM_TRAY_W}px — single class, "
                      f"each model in its native mode at the trained object scale",
               "S_3dtrab_downscaled.csv")
    print("\n  Compare S against B: same models, same annotations, same metric — the only")
    print("  difference is that the insects now arrive at the size the models were")
    print("  trained on. Any gain here is pure object-scale matching.")

dfUA = dfUB = None
if rowsUA:
    dfUA = show(rowsUA, f"U-A. SYNTHETIC — UNIFORM tiled inference, every model identical "
                        f"(normalised to {NORM_TRAY_W}px, tile {TILE})", "UA_synthetic_uniform.csv")
if rowsUB:
    dfUB = show(rowsUB, f"U-B. 3DTRAB — UNIFORM tiled inference, every model identical "
                        f"(normalised to {NORM_TRAY_W}px, tile {TILE})", "UB_3dtrab_uniform.csv")
if rowsUA or rowsUB:
    print("\n  U-tables are the SYMMETRIC comparison: identical inference for every")
    print("  model, at the object scale they were all trained on. A/B/C above are the")
    print("  as-deployed comparison, each model run the way it was trained.")

for (lab, tag), df in per_class.items():
    df.to_csv(OUT_DIR/f"perclass__{tag}__{safe(lab)}.csv", index=False)
print(f"\nCSVs written to {OUT_DIR}")
print(f"\n*_corr columns = predicted boxes scaled x{BOX_SCALE} about their centre. At 1.00\n"
      f"  rotated-canvas inflation baked into the synthetic labels. See the diagnostic cell.")


## 4b. Box-convention diagnostic  *(either kernel)*


In [ ]:
# ============================================================
# D. BOX-CONVENTION CHECK — confirming the box fix held
# ============================================================
# HISTORY. In August the generator rotated each cutout with expand=True and recorded
# insect_pil.size as the label without re-measuring the alpha channel. expand=True
# grows the canvas to hold the rotated rectangle and leaves the corners transparent,
# so every box was the CANVAS, not the insect — inflated by 4/pi = 1.273 per side.
# Models learned to predict inflated boxes and ran large against tightly-annotated
# real data; the sweep below peaked at x0.80.
#
# The September generator measures the box from the alpha channel after rotation, so
# this cell is now a CONFIRMATION rather than a diagnosis. What to expect:
#   D1  pred/GT diagonal near 1.0 rather than ~1.25
#   D2  the sweep peaking at x1.00, and "gain" near 1.0x
# A peak still near 0.80 would mean something in the pipeline is inflating boxes.
import numpy as np, pandas as pd

def _iou_mat(a, b):
    if len(a)==0 or len(b)==0: return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:,None,:2], b[None,:,:2]); rb = np.minimum(a[:,None,2:], b[None,:,2:])
    wh = np.clip(rb-lt, 0, None); inter = wh[...,0]*wh[...,1]
    aa = (a[:,2]-a[:,0])*(a[:,3]-a[:,1]); ab = (b[:,2]-b[:,0])*(b[:,3]-b[:,1])
    return inter/np.clip(aa[:,None]+ab[None]-inter, 1e-9, None)

_ann = json.load(open(TRAB_COCO))
_cname = {c["id"]: c["name"] for c in _ann["categories"]}
_N_GT = len(_ann["annotations"])
_GT_MED = float(np.median([np.hypot(a["bbox"][2], a["bbox"][3]) for a in _ann["annotations"]]))
_gt = {}
for a in _ann["annotations"]:
    x,y,w,h = a["bbox"]
    _gt.setdefault(a["image_id"], []).append((a["category_id"], [x,y,x+w,y+h]))

def diagnose(pred_file):
    """ceiling recall (any overlapping box, ignoring class AND confidence),
    plus how much bigger the matched predictions are than the annotations."""
    P = json.load(open(pred_file))["images"]
    ceil, ratios, ious, gsz, per_ord = [], [], [], [], {}
    for iid, items in _gt.items():
        G = np.array([t[1] for t in items], float); C = [t[0] for t in items]
        pr = P.get(str(iid))
        B = np.array(pr["boxes"], float).reshape(-1,4) if pr else np.zeros((0,4))
        if not len(B):
            ceil += [0.0]*len(G); continue
        M = _iou_mat(G, B); mx = M.max(1); j = M.argmax(1)
        ceil += list(mx >= 0.5)
        for i in range(len(G)):
            if mx[i] <= 0.1: continue
            gd = np.hypot(G[i,2]-G[i,0], G[i,3]-G[i,1])
            pd_ = np.hypot(B[j[i],2]-B[j[i],0], B[j[i],3]-B[j[i],1])
            ratios.append(pd_/gd); ious.append(mx[i]); gsz.append(gd)
            per_ord.setdefault(_cname[C[i]], []).append(pd_/gd)
    return (np.mean(ceil), np.median(ratios), np.median(ious), len(ratios),
            np.median(gsz) if gsz else 0.0, per_ord)

print("="*96)
print("D1. Are the insects being FOUND?  (ceiling recall ignores class and confidence)")
print("="*96)
rows, ords = [], {}
for lab, kind, w, mode, sz in _have:
    cr, rt, iou, n, gs, po = diagnose(pred_path(lab,"3dtrab"))
    rows.append({"model":lab,"mode":mode,"ceiling_recall@50":round(cr,3),
                 "pred/GT diagonal":round(rt,3),"median IoU of match":round(iou,3),
                 "matched":n,"of_GT":f"{n/_N_GT:.0%}","GT px (matched)":round(gs)})
    if kind == "rfdetr" and mode == "whole": ords[lab] = po
print(pd.DataFrame(rows).sort_values("ceiling_recall@50", ascending=False).to_string(index=False))
print("\n  A model whose ceiling recall is far above its mAP is finding insects and losing")
print("  them to the IoU threshold. One near zero is genuinely not firing.")
print(f"\n  CAUTION on 'pred/GT diagonal': it is measured over MATCHED pairs only, so it is")
print(f"  trustworthy where 'of_GT' is high and selection-biased where it is low. Compare")
print(f"  'GT px (matched)' against the {_GT_MED:.0f} px median over ALL {_N_GT} annotations —")
print(f"  a model that only matches the smaller insects reports a falsely healthy ratio.")

print("\n" + "="*96)
print(f"D2. Box-size sweep — mAP@50:95 on 3DTRAB with predictions scaled about their centre")
print("="*96)
# Symmetric about 1.00. The August range only went downward, because the models had
# been trained on inflated boxes and could only ever over-predict. With correct
# training boxes the error can go either way, and a one-sided sweep would report
# "best = x1.00" by default while hiding an optimum above it.
KS = [1.30, 1.20, 1.10, 1.00, 0.90, 0.85, 0.80, 0.75, 0.70]
sweep = []
for lab, kind, w, mode, sz in _have:
    r = {"model": lab}
    for k in KS:
        s,_ = evaluate(pred_path(lab,"3dtrab"), TRAB_COCO, single_class=True, box_scale=k)
        r[f"x{k:.2f}"] = round(s["mAP50_95"], 4)
    r["best"] = f"x{KS[int(np.argmax([r[f'x{k:.2f}'] for k in KS]))]:.2f}"
    r["gain"] = f"{max(r[f'x{k:.2f}'] for k in KS)/max(r['x1.00'],1e-9):.1f}x"
    sweep.append(r)
df_sweep = pd.DataFrame(sweep)
df_sweep.to_csv(OUT_DIR/"D_box_scale_sweep.csv", index=False)
print(df_sweep.to_string(index=False))
print(f"\n  Theory: uniform rotation inflates a box by E[|cos|+|sin|] = 4/pi = {4/np.pi:.3f}")
print(f"  for a round body. The September ground truth is already correct, so the\n"
        f"  optimum here should be ~1.00. A peak near 0.80 would mean boxes are still\n"
        f"  inflated somewhere in the pipeline.")

print("\n" + "="*96)
print("D3. Per-order box-size ratio — predicted vs annotated, by order")
print("="*96)
for lab, po in ords.items():
    r = [{"order":k, "n":len(v), "pred/GT diagonal":round(float(np.median(v)),3)}
         for k,v in po.items() if len(v) >= 20]
    if not r: continue
    print(f"\n{lab}:")
    print(pd.DataFrame(r).sort_values("pred/GT diagonal", ascending=False).to_string(index=False))
# Report what the data says; do NOT restate the August conclusion. In August this
# table distinguished two explanations for inflated boxes — rotation geometry (round
# bodies worst, elongated best) versus annotators including appendages (long-legged
# orders worst). Orthoptera came out lowest, which rejected the appendage explanation.
# With the boxes fixed the question is settled and every order should now sit near 1.0.
_allr = [v for po in ords.values() for vs in po.values() for v in vs]
if _allr:
    _med = float(np.median(_allr))
    print(f"\n  Median across all orders: {_med:.3f}")
    if 0.9 <= _med <= 1.1:
        print("  Near 1.0 — predictions match the annotation convention. The August box")
        print("  inflation (~1.25) is gone.")
    else:
        print(f"  NOT near 1.0. Predictions run {'large' if _med > 1 else 'small'} against")
        print("  the real annotations. Check D2's optimum before trusting the mAP figures.")
    _spread = {k: float(np.median(v)) for po in ords.values() for k, v in po.items() if len(v) >= 20}
    if len(_spread) >= 3:
        _lo = min(_spread, key=_spread.get); _hi = max(_spread, key=_spread.get)
        print(f"  Spread: {_lo} {_spread[_lo]:.3f} (lowest) to {_hi} {_spread[_hi]:.3f} (highest).")
        print("  A large spread means the offset is order-specific rather than systematic.")
else:
    print("\n  (no whole-tray RF-DETR predictions cached yet — D3 needs them)")


## 4c. Confidence threshold — does a lower cut-off find more insects?  *(either kernel)*


In [ ]:
# ============================================================
# T. CONFIDENCE THRESHOLD — does a lower cut-off find more insects?
# ============================================================
# Two different things get called "the threshold", and only one of them can
# change how much a model detects:
#
#  * the SCORING threshold picks an operating point on a curve that already
#    exists. It moves precision and recall against each other. It cannot change
#    mAP, because mAP already integrates over every prediction.
#  * the INFERENCE floor (CONF_EVAL) decides which boxes get emitted at all.
#    Anything below it was never produced, so it is a hard ceiling on recall.
#
# T1 sweeps the first from the cached predictions (free). T2 scores the low-floor
# pass, which is the only way to genuinely detect more.
import pandas as pd
from collections import defaultdict

_t_ann = json.load(open(TRAB_DS_COCO))
_t_gt  = defaultdict(list)
for _a in _t_ann["annotations"]:
    x, y, w, h = _a["bbox"]
    _t_gt[_a["image_id"]].append([x, y, x + w, y + h])
_T_TOTAL = sum(len(v) for v in _t_gt.values())

def operating_point(P, conf_thr, iou_thr=0.5):
    """Class-agnostic match at iou_thr, after the same NMS the single-class view
    uses — so these numbers sit on the same footing as table B."""
    matched = n_pred = 0
    for iid, g in _t_gt.items():
        pr = P.get(str(iid))
        if not pr: continue
        b = np.array(pr["boxes"], float).reshape(-1, 4)
        s = np.array(pr["scores"], float)
        if len(b):
            k = s >= conf_thr
            b, s = b[k], s[k]
        if len(b):
            keep = _agnostic_nms(b, s, IOU_NMS)
            b, s = b[keep], s[keep]
        n_pred += len(b)
        if len(b) and len(g):
            M = iou_mat(np.asarray(g, float), b)
            gi, pi = np.where(M >= iou_thr)
            if len(gi):
                o = np.argsort(-M[gi, pi]); ug, up = set(), set()
                for gg, pp in zip(gi[o], pi[o]):
                    if gg in ug or pp in up: continue
                    ug.add(int(gg)); up.add(int(pp))
                matched += len(ug)
    rec = matched / max(_T_TOTAL, 1)
    prec = matched / max(n_pred, 1)
    return dict(found=matched, missed=_T_TOTAL - matched, n_pred=n_pred,
                recall=round(rec, 4), precision=round(prec, 4),
                F1=round(2 * prec * rec / max(prec + rec, 1e-9), 4))

THRS = [0.005, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

print("=" * 100)
print(f"T1. OPERATING POINT vs CONFIDENCE — 3DTRAB downscaled, {_T_TOTAL} insects, IoU 0.50")
print("=" * 100)
_rows = []
for lab, kind, w, mode, sz in MODELS:
    for tag, floor in (("3dtrab_ds", CONF_EVAL),
                       ("3dtrab_ds_low", CONF_EVAL_LOW if LOW_CONF_PASS else None)):
        f = pred_path(lab, tag)
        if floor is None or not f.exists(): continue
        P = json.load(open(f))["images"]
        for t in THRS:
            if t < floor - 1e-9: continue          # below the floor nothing was emitted
            r = operating_point(P, t)
            r.update(model=lab, floor=floor, conf=t)
            _rows.append(r)
_sweep = pd.DataFrame(_rows)
_sweep.to_csv(OUT_DIR / "T_threshold_sweep.csv", index=False)

for lab, kind, w, mode, sz in MODELS:
    d = _sweep[_sweep.model == lab]
    if not len(d): continue
    print(f"\n{lab}")
    for floor in sorted(d["floor"].unique()):
        dd = d[d["floor"] == floor].sort_values("conf")
        best = dd.loc[dd["F1"].idxmax()]
        print(f"  inference floor {floor}:")
        print("   " + dd[["conf","found","missed","n_pred","recall","precision","F1"]]
              .to_string(index=False).replace("\n", "\n   "))
        print(f"    best F1 at conf {best['conf']}: finds {int(best['found'])}"
              f"/{_T_TOTAL} insects, {int(best['n_pred'])} predictions")

print("\n" + "=" * 100)
print("T2. FULL METRICS AT THE LOWER FLOOR — this is the part that can change mAP")
print("=" * 100)
_r2 = []
for lab, kind, w, mode, sz in MODELS:
    hi, lo = pred_path(lab, "3dtrab_ds"), pred_path(lab, "3dtrab_ds_low")
    if not hi.exists(): continue
    a, _ = evaluate(hi, TRAB_DS_COCO, single_class=True)
    row = {"model": lab, "floor": CONF_EVAL, "mAP50_95": round(a["mAP50_95"], 4),
           "mAP50": round(a["mAP50"], 4), "mAP75": round(a["mAP75"], 4),
           "P": round(a["precision"], 3), "R": round(a["recall"], 3), "n_pred": a["n_pred"]}
    _r2.append(row)
    if lo.exists():
        b, _ = evaluate(lo, TRAB_DS_COCO, single_class=True)
        _r2.append({"model": lab, "floor": CONF_EVAL_LOW,
                    "mAP50_95": round(b["mAP50_95"], 4), "mAP50": round(b["mAP50"], 4),
                    "mAP75": round(b["mAP75"], 4), "P": round(b["precision"], 3),
                    "R": round(b["recall"], 3), "n_pred": b["n_pred"],
                    "d_mAP": round(b["mAP50_95"] - a["mAP50_95"], 4),
                    "d_recall": round(b["recall"] - a["recall"], 3)})
if _r2:
    _df2 = pd.DataFrame(_r2)
    _df2.to_csv(OUT_DIR / "T_low_floor.csv", index=False)
    print(_df2.to_string(index=False))
    if "d_mAP" in _df2.columns and _df2["d_mAP"].notna().any():
        _g = _df2["d_mAP"].dropna()
        print(f"\n  mAP change from dropping the floor to {CONF_EVAL_LOW}: "
              f"median {_g.median():+.4f}, best {_g.max():+.4f}, worst {_g.min():+.4f}")
        print("  A gain means the floor was throwing away real detections.")
        print("  A flat or negative result means the model had already emitted everything")
        print("  it could find, and the extra boxes are noise — recall is capped by the")
        print("  model, not by the threshold. Compare against ceiling recall in D1.")
else:
    print("  no predictions found — run the RUN cells first")
if not LOW_CONF_PASS:
    print(f"\n  LOW_CONF_PASS is False: set it True in CONFIG and re-run the RUN cells")
    print(f"  to produce the {CONF_EVAL_LOW} pass. T1 above still shows every operating")
    print(f"  point at or above the current floor of {CONF_EVAL}.")


## 5. Figures


In [ ]:
# ── Figures ──
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PALETTE = ["#2563eb","#16a34a","#dc2626","#d97706","#7c3aed","#0891b2","#be185d","#4d7c0f"]
def _cols(n): return [PALETTE[i % len(PALETTE)] for i in range(n)]
def _save(fig, fname):
    fig.tight_layout(); fig.savefig(OUT_DIR/fname, dpi=140, bbox_inches="tight"); plt.show()

def barfig(df, title, fname, keys=("mAP50_95","mAP50","mAP75")):
    keys = [k for k in keys if k in df.columns]
    labs = df["model"].tolist(); x = np.arange(len(keys)); w = 0.8/max(len(labs),1)
    cols = _cols(len(labs))
    fig, ax = plt.subplots(figsize=(max(10, 1.6*len(labs)+6), 6))
    for j,l in enumerate(labs):
        v = [df[df["model"]==l][k].values[0] for k in keys]
        b = ax.bar(x + (j-(len(labs)-1)/2)*w, v, w, label=l, color=cols[j])
        ax.bar_label(b, fmt="%.3f", fontsize=7, padding=2)
    ax.set_xticks(x)
    ax.set_xticklabels([{"mAP50_95":"mAP@50:95","mAP50":"mAP@50","mAP75":"mAP@75"}.get(k,k) for k in keys])
    ax.set_ylabel("score"); ax.set_title(title, fontsize=13)
    ax.grid(axis="y", alpha=.3); ax.legend(fontsize=8, ncol=2); ax.set_ylim(0, None)
    _save(fig, fname)

def pairfig(labs, left, right, lname, rname, title, ylabel, fname, annotate=True):
    """Two bars per model — the workhorse for before/after comparisons."""
    xs = np.arange(len(labs)); w = 0.38
    fig, ax = plt.subplots(figsize=(max(9, 1.5*len(labs)+4), 6))
    b1 = ax.bar(xs-w/2, left,  w, label=lname, color="#94a3b8")
    b2 = ax.bar(xs+w/2, right, w, label=rname, color="#2563eb")
    if annotate:
        ax.bar_label(b1, fmt="%.3f", fontsize=7, padding=2)
        ax.bar_label(b2, fmt="%.3f", fontsize=7, padding=2)
        for i,(a,b) in enumerate(zip(left,right)):
            if a > 1e-9:
                ax.annotate(f"{b/a:.1f}x", (xs[i], max(a,b)), textcoords="offset points",
                            xytext=(0,16), ha="center", fontsize=8, color="#16a34a", weight="bold")
    ax.set_xticks(xs); ax.set_xticklabels(labs, rotation=25, ha="right", fontsize=9)
    ax.set_ylabel(ylabel); ax.set_title(title, fontsize=13)
    ax.grid(axis="y", alpha=.3); ax.legend()
    top = max(list(left) + list(right) + [1e-6])
    ax.set_ylim(0, top * (1.22 if annotate else 1.10))   # room for the labels
    _save(fig, fname)

# ── 1-3. per-view bars (as-deployed: each model run the way it was trained) ──
barfig(dfA, "A. Synthetic test (22 classes)",           "A_synthetic.png")
barfig(dfB, "B. 3DTRAB real trays, single class",       "B_3dtrab_singleclass.png")
barfig(dfC, "C. 3DTRAB real trays, 5 annotated orders", "C_3dtrab_5class.png")

# ── 4. the headline gap ──
labs = dfA["model"].tolist()
pairfig(labs,
        [float(dfA[dfA.model==l].mAP50_95.values[0]) for l in labs],
        [float(dfB[dfB.model==l].mAP50_95.values[0]) for l in labs],
        "synthetic (22-class)", "3DTRAB real (single-class)",
        "Synthetic vs real — the sim-to-real gap", "mAP@50:95",
        "D_synthetic_vs_real.png", annotate=False)

# ── 5. how much of the real-data gap is the rotated-canvas label bug ──
if "mAP50_95_corr" in dfB.columns:
    pairfig(labs,
            [float(dfB[dfB.model==l].mAP50_95.values[0]) for l in labs],
            [float(dfB[dfB.model==l].mAP50_95_corr.values[0]) for l in labs],
            "raw", f"boxes x{BOX_SCALE}",
            f"E. 3DTRAB — correcting the rotated-canvas box inflation", "mAP@50:95",
            "E_box_correction.png")

# ── 5b. full-resolution vs downscaled to the trained object scale ──
if dfS is not None:
    sl = [l for l in labs if l in set(dfS["model"])]
    pairfig(sl,
            [float(dfB[dfB.model==l].mAP50_95.values[0]) for l in sl],
            [float(dfS[dfS.model==l].mAP50_95.values[0]) for l in sl],
            "full resolution (3473px)", f"downscaled to {NORM_TRAY_W}px",
            "S. 3DTRAB — matching the object scale the models were trained on",
            "mAP@50:95", "S_downscaled.png")

# ── 6. as-deployed vs identical inference for every model ──
if dfUB is not None:
    ul = [l for l in labs if l in set(dfUB["model"])]
    pairfig(ul,
            [float(dfB[dfB.model==l].mAP50_95.values[0]) for l in ul],
            [float(dfUB[dfUB.model==l].mAP50_95.values[0]) for l in ul],
            "as trained", f"uniform tiled @{NORM_TRAY_W}px",
            "F. 3DTRAB — native inference vs identical tiled inference", "mAP@50:95",
            "F_uniform_3dtrab.png")
if dfUA is not None:
    ul = [l for l in labs if l in set(dfUA["model"])]
    pairfig(ul,
            [float(dfA[dfA.model==l].mAP50_95.values[0]) for l in ul],
            [float(dfUA[dfUA.model==l].mAP50_95.values[0]) for l in ul],
            "as trained", f"uniform tiled @{NORM_TRAY_W}px",
            "G. Synthetic — native inference vs identical tiled inference", "mAP@50:95",
            "G_uniform_synthetic.png")

# ── 7. precision/recall on real trays: where each model actually operates ──
fig, ax = plt.subplots(figsize=(8,6.5))
for j,l in enumerate(labs):
    r = dfB[dfB.model==l]
    ax.scatter(r["recall"], r["precision"], s=150, color=_cols(len(labs))[j],
               edgecolor="white", linewidth=1.5, zorder=3, label=l)
for f1 in (0.1,0.2,0.3,0.4,0.5):
    x = np.linspace(0.01,1,200); y = f1*x/(2*x-f1)
    ok = (y>0)&(y<=1)
    ax.plot(x[ok], y[ok], color="#cbd5e1", lw=.9, zorder=1)
    if ok.any():
        ax.annotate(f"F1={f1}", (x[ok][-1], y[ok][-1]), fontsize=7, color="#94a3b8",
                    textcoords="offset points", xytext=(-22, 4))
ax.set_xlabel("recall"); ax.set_ylabel("precision")
ax.set_title("H. 3DTRAB real trays — operating point (best-F1, IoU 0.50)", fontsize=13)
ax.set_xlim(0,None); ax.set_ylim(0,None); ax.grid(alpha=.3); ax.legend(fontsize=8)
_save(fig, "H_precision_recall.png")

# ── 8. the box-scale sweep, if the diagnostic cell has run ──
_sw = OUT_DIR/"D_box_scale_sweep.csv"
if _sw.exists():
    sw = pd.read_csv(_sw)
    kcols = [c for c in sw.columns if c.startswith("x")]
    ks = [float(c[1:]) for c in kcols]
    fig, ax = plt.subplots(figsize=(9,6))
    for j,r in sw.iterrows():
        ax.plot(ks, [r[c] for c in kcols], marker="o", color=_cols(len(sw))[j], label=r["model"])
    ax.axvline(BOX_SCALE, color="#dc2626", ls="--", lw=1,
               label=f"no correction due (x{BOX_SCALE})")
    ax.invert_xaxis()
    ax.set_xlabel("predicted boxes scaled about their centre")
    ax.set_ylabel("mAP@50:95"); ax.grid(alpha=.3); ax.legend(fontsize=8)
    ax.set_title("I. 3DTRAB — box-size sweep (peak reveals the label inflation)", fontsize=13)
    _save(fig, "I_box_scale_sweep.png")

print("figures written to", OUT_DIR)
for f in sorted(OUT_DIR.glob("*.png")): print("  ", f.name)


## 6. Visual check — ground truth vs predictions on real trays


In [ ]:
VIS_COCO = TRAB_DS_COCO      # the downscaled set: where every model scores best
VIS_IMGS = TRAB_DS_IMGS

# ============================================================
# VISUAL CHECK — ground truth vs predictions on real trays
# ============================================================
# A number says a model scores 0.07. This says *how* it fails: whether it misses
# insects, doubles up on them, or finds them and boxes them badly.
#
#   green   a ground-truth insect this model found      (true positive)
#   red     a ground-truth insect it missed             (false negative)
#   orange  a prediction matching no insect             (false positive)
#
# Matching is class-agnostic at IoU 0.5 — the same rule as the single-class table,
# so the panels and the numbers agree.
import matplotlib
matplotlib.use("Agg")
import re
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from collections import defaultdict
from PIL import Image

VIS_CONF   = 0.30     # display threshold; AP uses the full ranking, a picture cannot
VIS_TRAYS  = 3        # how many trays to draw
VIS_IOU    = 0.50

_vis_ann  = json.load(open(VIS_COCO))
_vis_meta = {i["id"]: i for i in _vis_ann["images"]}
_vis_gt   = defaultdict(list)
for _a in _vis_ann["annotations"]:
    x, y, w, h = _a["bbox"]
    _vis_gt[_a["image_id"]].append([x, y, x + w, y + h])

def _match(gt, pred, thr=VIS_IOU):
    """Greedy class-agnostic matching. -> (matched_gt, missed_gt, fp_pred) index sets."""
    gt = np.asarray(gt, float).reshape(-1, 4); pred = np.asarray(pred, float).reshape(-1, 4)
    if len(gt) == 0 or len(pred) == 0:
        return set(), set(range(len(gt))), set(range(len(pred)))
    M = iou_mat(pred, gt)
    pi, gi = np.where(M >= thr)
    o = np.argsort(-M[pi, gi])
    up, ug = set(), set()
    for p_, g_ in zip(pi[o], gi[o]):
        if p_ in up or g_ in ug: continue
        up.add(int(p_)); ug.add(int(g_))
    return ug, set(range(len(gt))) - ug, set(range(len(pred))) - up

def _panel(ax, img, gt, pred, title):
    ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
    if pred is None:                                  # ground-truth reference panel
        for b in gt:
            ax.add_patch(Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                                   fill=False, edgecolor="#16a34a", lw=1.1))
        ax.set_title(f"{title}\n{len(gt)} annotated", fontsize=9)
        return
    ok, miss, fp = _match(gt, pred)
    for i, b in enumerate(gt):
        c = "#16a34a" if i in ok else "#dc2626"
        ax.add_patch(Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                               fill=False, edgecolor=c, lw=1.1))
    for i, b in enumerate(pred):
        if i not in fp: continue
        ax.add_patch(Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                               fill=False, edgecolor="#d97706", lw=0.9, ls=(0, (3, 2))))
    p = len(ok)/max(len(pred), 1); r = len(ok)/max(len(gt), 1)
    ax.set_title(f"{title}\nfound {len(ok)}/{len(gt)}  extra {len(fp)}  "
                 f"P {p:.2f} R {r:.2f}", fontsize=9)

def _load_preds(path):
    if path is None or not Path(path).exists(): return None
    return json.load(open(path))["images"]

def visualise(panels, image_ids, fname, suptitle):
    """panels: list of (column title, predictions-dict or None)."""
    ncol = len(panels)
    fig, axes = plt.subplots(len(image_ids), ncol,
                             figsize=(4.6*ncol, 3.6*len(image_ids)), squeeze=False)
    for r, iid in enumerate(image_ids):
        meta = _vis_meta[iid]
        with Image.open(VIS_IMGS / meta["file_name"]) as im:
            img = im.convert("RGB").copy()
        gt = _vis_gt[iid]
        for c, (title, P) in enumerate(panels):
            pred = None
            if P is not None:
                d = P.get(str(iid), {"boxes": [], "scores": []})
                b = np.array(d["boxes"], float).reshape(-1, 4)
                s = np.array(d["scores"], float)
                pred = b[s >= VIS_CONF] if len(b) else b
            _panel(axes[r][c], img, gt, pred,
                   title if r == 0 else f"{title}")
        axes[r][0].set_ylabel(Path(meta["file_name"]).stem[:26], fontsize=8)
    fig.legend(handles=[Line2D([], [], color="#16a34a", lw=2, label="found"),
                        Line2D([], [], color="#dc2626", lw=2, label="missed"),
                        Line2D([], [], color="#d97706", lw=2, ls="--", label="false positive")],
               loc="lower center", ncol=3, frameon=False, fontsize=9)
    fig.suptitle(f"{suptitle}   (predictions shown at confidence >= {VIS_CONF})",
                 fontsize=12)
    fig.tight_layout(rect=[0, 0.035, 1, 0.97])
    fig.savefig(OUT_DIR / fname, dpi=130, bbox_inches="tight")
    plt.show()
    print("wrote", OUT_DIR / fname)

def _grp(fn):
    n = Path(fn).name
    m = re.match(r"Overview_(\d*D\d+)_", n)
    if m: return m.group(1)
    m = re.match(r"Sample_(\d+)_", n)
    return "Sample_" + m.group(1) if m else n

# ── choose trays: spread over groups, and pick typical rather than extreme frames ──
_by_group = defaultdict(list)
for iid, m in _vis_meta.items():
    _by_group[_grp(m["file_name"])].append(iid)
_pick = []
for g in sorted(_by_group, key=lambda g: -sum(len(_vis_gt[i]) for i in _by_group[g])):
    ids = sorted(_by_group[g], key=lambda i: len(_vis_gt[i]))
    _pick.append(ids[len(ids)//2])            # median-density frame of that group
    if len(_pick) >= VIS_TRAYS: break
print("trays:", [Path(_vis_meta[i]["file_name"]).stem for i in _pick],
      "| insects:", [len(_vis_gt[i]) for i in _pick])

panels = [("ground truth", None)]
for lab, kind, w, mode, sz in MODELS:
    panels.append((lab, _load_preds(pred_path(lab, "3dtrab_ds"))))
visualise(panels, _pick, "VIS_all_models.png",
          "3DTRAB real trays - ground truth vs each model")


---
## K. Confusion — detection failure or naming failure?

Per-class AP says a class scores badly. It cannot say whether the detector **never
found** the insect, **found it and named it wrong**, or **hallucinated it** where
nothing was. Those three have different remedies — more specimens, more separable
classes, or a higher threshold — and telling them apart is what turns a bad number
into an action.

Computed on the synthetic test split, the only set carrying all 22 classes in
quantity. Writes a full 22x23 matrix per model plus a ranked summary.


In [ ]:
# ============================================================
# K. CONFUSION — which class does each model mistake for which?
# ============================================================
# mAP tells you a class scores badly. It does not tell you WHY. A class can fail
# because the detector never finds it (missed), because it finds it and names it
# something else (confused), or because it hallucinates it where nothing is
# (false positive). Those have different fixes, and the per-class AP tables cannot
# distinguish them.
#
# Computed on the SYNTHETIC test split, which is the only set with all 22 classes
# present in quantity. Each ground-truth box is matched to its best-IoU prediction
# above CONF_MAT_CONF; the outcome is the predicted class, or "missed".
CONF_MAT_IOU  = 0.50
CONF_MAT_CONF = 0.30

_ann  = json.load(open(SYN_COCO / SYN_SPLIT / "_annotations.coco.json"))
_cats = sorted(_ann["categories"], key=lambda c: c["id"])
_cname = [c["name"] for c in _cats]
_cidx  = {c["id"]: k for k, c in enumerate(_cats)}
_gt = {}
for a in _ann["annotations"]:
    x, y, w, h = a["bbox"]
    _gt.setdefault(a["image_id"], []).append((_cidx[a["category_id"]], x, y, x+w, y+h))

_NC = len(_cname)
_conf_rows = []
for lab, kind, w_, mode, sz in _have:
    _pf = pred_path(lab, "synthetic")
    if not Path(_pf).exists():
        print(f"  {lab}: no predictions, skipped"); continue
    P = json.load(open(_pf))["images"]
    M = np.zeros((_NC, _NC + 1), float)      # last column = missed
    FP = np.zeros(_NC, float)                # predicted where no GT
    for iid, g in _gt.items():
        pr = P.get(str(iid)) or P.get(int(iid)) or None
        gb = np.array([[a[1], a[2], a[3], a[4]] for a in g], float)
        gc = np.array([a[0] for a in g], int)
        if not pr or not len(pr["boxes"]):
            for c in gc: M[c, _NC] += 1
            continue
        pb = np.array(pr["boxes"], float).reshape(-1, 4)
        ps = np.array(pr["scores"], float); pc = np.array(pr["classes"], int)
        k = ps >= CONF_MAT_CONF
        pb, ps, pc = pb[k], ps[k], pc[k]
        if not len(pb):
            for c in gc: M[c, _NC] += 1
            continue
        I = iou_mat(gb, pb)
        used = set()
        for gi in np.argsort(-gb[:, 2] * gb[:, 3]):        # big boxes claim first
            order = np.argsort(-I[gi])
            hit = next((int(j) for j in order
                        if I[gi, j] >= CONF_MAT_IOU and j not in used), None)
            if hit is None: M[gc[gi], _NC] += 1
            else:           M[gc[gi], pc[hit]] += 1; used.add(hit)
        for j in range(len(pb)):
            if j not in used and 0 <= pc[j] < _NC: FP[pc[j]] += 1

    _tot = M.sum(1)
    for c in range(_NC):
        if _tot[c] == 0: continue
        _correct = M[c, c] / _tot[c]
        _missed  = M[c, _NC] / _tot[c]
        _wrong   = 1.0 - _correct - _missed
        _off = [(M[c, k] / _tot[c], _cname[k]) for k in range(_NC) if k != c and M[c, k] > 0]
        _off.sort(reverse=True)
        _conf_rows.append({
            "model": lab, "class": _cname[c], "instances": int(_tot[c]),
            "correct": round(_correct, 4), "wrong_class": round(_wrong, 4),
            "missed": round(_missed, 4),
            "top_confusion": f"{_off[0][1]} {_off[0][0]:.0%}" if _off else "",
            "2nd_confusion": f"{_off[1][1]} {_off[1][0]:.0%}" if len(_off) > 1 else "",
            "false_pos": int(FP[c]),
        })
    np.savetxt(OUT_DIR / f"K_confusion__{safe(lab)}.csv",
               np.c_[M, FP], delimiter=",", fmt="%d",
               header=",".join(_cname + ["missed", "false_pos"]), comments="")

_cdf = pd.DataFrame(_conf_rows)
if len(_cdf):
    _cdf.to_csv(OUT_DIR / "K_confusion_summary.csv", index=False)
    print(f"K. CONFUSION BREAKDOWN — synthetic test, IoU>={CONF_MAT_IOU}, conf>={CONF_MAT_CONF}\n")
    for lab in _cdf.model.unique():
        d = _cdf[_cdf.model == lab].sort_values("correct")
        print(f"  {lab}   (worst 8 classes by correct-rate)")
        print("   " + d.head(8).drop(columns=["model"]).to_string(index=False).replace("\n", "\n   "))
        _cf = d[d.wrong_class > d.missed]
        print(f"   -> confused more than missed: "
              f"{', '.join(_cf['class'].tolist()) if len(_cf) else 'none'}\n")
    print(f"  full matrices -> {OUT_DIR}/K_confusion__<model>.csv")
    print("  A class with a high 'missed' rate is a DETECTION failure — the model never")
    print("  found it. A class with a high 'wrong_class' rate is a NAMING failure — it")
    print("  found the insect and called it something else. Those need different fixes:")
    print("  more or better specimens for the first, more separable classes for the second.")
else:
    print("no predictions available — run the scoring cell first")
